## **Ingredient Price Supplementation**

For the web application, ingredient data needs to be supplemented with price information. 

This notebook does so using an LLM to query the price of each ingredient for an average Maltese consumer in 2026.

In [1]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

load_dotenv()

True

## Environment Setup

In [2]:
with open("./supplemented_structured_ingredients.json", "r") as file:
    ingredients = json.load(file)

In [3]:
ingredient_id_to_name = {ingredient["id"]: ingredient["name"] for ingredient in ingredients}

In [4]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
BATCH_SIZE = 100
MODEL_NAME = "gpt-5.4-mini"

PROMPT_TEMPLATE = """\
You are a food pricing assistant. Your task is to estimate the retail price per 100g in EUR (€) for each of the following ingredients.

Use your knowledge of typical Maltese supermarket prices, estimating the price in 2026. For uncommon ingredients, make a reasonable estimate.

Return ONLY a valid JSON object in this exact format, with no other text:
{{{{"<ingredient_id>": <price_per_100g_in_eur>, ...}}

Ingredients will be provided in the following format:
{{"<ingredient_id>": "<ingredient_name>", ...}}

Ingredients:
{ingredient_list}
"""

In [6]:
print(PROMPT_TEMPLATE.format(ingredient_list={"test_id": "test_name", "test_id_2": "test_name_2"}))

You are a food pricing assistant. Your task is to estimate the retail price per 100g in EUR (€) for each of the following ingredients.

Use your knowledge of typical Maltese supermarket prices, estimating the price in 2026. For uncommon ingredients, make a reasonable estimate.

Return ONLY a valid JSON object in this exact format, with no other text:
{{"<ingredient_id>": <price_per_100g_in_eur>, ...}

Ingredients will be provided in the following format:
{"<ingredient_id>": "<ingredient_name>", ...}

Ingredients:
{'test_id': 'test_name', 'test_id_2': 'test_name_2'}



In [7]:
llm_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Price Estimation with LLMs

In [8]:
def estimate_prices_for_batch(ingredient_id_to_name: dict[str, str]) -> dict[str, float]:
    """
    Sends a batch of ingredient names to the LLM and returns a dict mapping each name to its estimated price per 100g in EUR

    :param ingredient_id_to_name: dict mapping ingredient IDs to names
    :type ingredient_id_to_name: dict[str, str]

    :return: dict mapping ingredient IDs to estimated prices per 100g in EUR
    :rtype: dict[str, float]
    """

    prompt = PROMPT_TEMPLATE.format(ingredient_list=ingredient_id_to_name)

    response = llm_client.responses.create(model=MODEL_NAME, input=prompt)

    prices = response.output[0].content[0].text

    return json.loads(prices)

In [9]:
ingredient_id_to_price = {}

In [ ]:
for batch in tqdm(
    [
        dict(list(ingredient_id_to_name.items())[i : i + BATCH_SIZE])
        for i in range(0, len(ingredient_id_to_name), BATCH_SIZE)
    ],
    desc="Estimating prices",
):
    batch_prices = estimate_prices_for_batch(batch)
    ingredient_id_to_price.update(batch_prices)

Estimating prices:   0%|          | 0/161 [00:00<?, ?it/s]

## Exporting the Extracted Data

In [11]:
with open("./priced_ingredients.json", "w") as file:
    json.dump(ingredient_id_to_price, file, indent=4)